In [1]:
!pip install faiss-cpu chromadb sentence-transformers pandas tqdm


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import pandas as pd
from tqdm.auto import tqdm
import faiss
import chromadb
from sentence_transformers import SentenceTransformer


In [3]:
DATASET_DIR = r'c:\Users\kaushal\Desktop\major 2\dataset'
RAG_CHUNKS_FILE = os.path.join(DATASET_DIR, 'rag_chunks', 'all_chunks.jsonl')
STRUCTURED_DIR = os.path.join(DATASET_DIR, 'structured')

documents = []
metadatas = []
ids = []

# 1. Load Pre-chunked RAG data
print(f"Loading data from {RAG_CHUNKS_FILE}...")
if os.path.exists(RAG_CHUNKS_FILE):
    with open(RAG_CHUNKS_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            data = json.loads(line)
            text = data.get('text', '')
            chunk_id = data.get('chunk_id', f'chunk_{i}')
            if text:
                documents.append(text)
                metadata = {k: str(v) if v is not None else '' for k, v in data.items() if k != 'text'}
                metadatas.append(metadata)
                ids.append(chunk_id)

print(f"Loaded {len(documents)} documents from all_chunks.jsonl")

# 2. Optionally load Structured JSONs
print(f"Scanning for structured JSON files in {STRUCTURED_DIR}...")
for root, dirs, files in os.walk(STRUCTURED_DIR):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        for idx, item in enumerate(data):
                            text = item.get('text', '')
                            if text:
                                documents.append(text)
                                metadata = {k: str(v) if v is not None else '' for k, v in item.items() if k != 'text'}
                                metadata['source_file'] = file_path
                                metadatas.append(metadata)
                                ids.append(f"{file}_{idx}")
                except json.JSONDecodeError:
                    print(f"Error decoding JSON from {file_path}")

print(f"Total documents loaded: {len(documents)}")


Loading data from c:\Users\kaushal\Desktop\major 2\dataset\rag_chunks\all_chunks.jsonl...
Loaded 5388 documents from all_chunks.jsonl
Scanning for structured JSON files in c:\Users\kaushal\Desktop\major 2\dataset\structured...


Total documents loaded: 13917


In [4]:
# Initialize embedding model
print("Initializing all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding documents...")
embeddings = model.encode(documents, batch_size=32, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")


Initializing all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding documents...


Batches:   0%|          | 0/435 [00:00<?, ?it/s]

Embeddings shape: (13917, 384)


In [5]:
# Initialize FAISS HNSW Index
print("Initializing FAISS HNSW index...")
dimension = embeddings.shape[1]
faiss_index = faiss.IndexHNSWFlat(dimension, 32)

print("Adding embeddings to FAISS...")
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'faiss_hnsw_index.bin')
print(f"FAISS index built with {faiss_index.ntotal} vectors.")


Initializing FAISS HNSW index...
Adding embeddings to FAISS...


FAISS index built with 13917 vectors.


In [6]:
# Initialize Chroma DB HNSW Index
print("Initializing Chroma DB...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection(name="rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_collection", 
    metadata={"hnsw:space": "cosine"}
)

print("Adding documents to Chroma DB (this might take a minute)...")
batch_size = 5000
for i in tqdm(range(0, len(documents), batch_size)):
    collection.add(
        documents=documents[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )

print(f"Chroma DB collection created with {collection.count()} vectors.")


Initializing Chroma DB...


Adding documents to Chroma DB (this might take a minute)...


  0%|          | 0/3 [00:00<?, ?it/s]

Chroma DB collection created with 13917 vectors.


In [7]:
# Testing both databases
query = "What are the phases of the author's relationship with his grandmother?"
query_embedding = model.encode([query])

print("--- FAISS Results ---")
D, I = faiss_index.search(query_embedding, k=3)
for idx, (dist, i) in enumerate(zip(D[0], I[0])):
    print(f"Rank {idx+1} (Dist: {dist:.4f}): {documents[i][:150]}...")

print("\n--- Chroma DB Results ---")
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)
for idx, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"Rank {idx+1}: {doc[:150]}...")


--- FAISS Results ---
Rank 1 (Dist: 0.5318): Question: The three phases of the author’s relationship with his grandmother before he left the country to study abroad.
Answer: The three phases of t...
Rank 2 (Dist: 0.7028): Question: Describe the changing relationship between the author and his grandmother. Did their feelings for each other change?
Answer: The author was ...
Rank 3 (Dist: 0.8474): Question: The odd way in which the author’s grandmother behaved just before she died.
Answer: The grandmother of the author did not speak to them befo...

--- Chroma DB Results ---
Rank 1: Question: The three phases of the author’s relationship with his grandmother before he left the country to study abroad.
Answer: The three phases of t...
Rank 2: Question: Describe the changing relationship between the author and his grandmother. Did their feelings for each other change?
Answer: The author was ...
Rank 3: Question: The odd way in which the author’s grandmother behaved just before she died.
An